# PatchCore ViT-B/16 (x64)

Mirrors the main ViT-B/16 PatchCore experiment at 64x64 resolution to complete the resolution
comparison table (WRN50 x64, EffNet-B0 x64 vs. ViT-B/16 x64 vs. ViT-B/16 x224).

At 64x64 the 16-pixel patch size yields (64/16) = 16 patch tokens per image
instead of 196. The positional embeddings are resized automatically by timm.

All other settings (block 6, 95th-pct val-normal threshold, topk=0.10, mb=400k) are
identical to the main x224 run.

Artifacts written by this notebook:
- `experiments/anomaly_detection/patchcore/vit_b16/x64/main/artifacts/checkpoints/`
- `experiments/anomaly_detection/patchcore/vit_b16/x64/main/artifacts/plots/`
- `experiments/anomaly_detection/patchcore/vit_b16/x64/main/artifacts/results/`


## Setup


In [ ]:
from importlib.util import find_spec
import sys
deps = [('timm', 'timm'), ('tqdm', 'tqdm'), ('scikit-learn', 'sklearn')]
missing = [pkg for pkg, module_name in deps if find_spec(module_name) is None]
DEPS_READY = not missing
if DEPS_READY:
    print('deps ready')
else:
    print(f'[WARNING] Missing dependencies: {missing}. This notebook run will skip dependency-backed cells.')


## Imports


In [ ]:
import gc, json, os, random, warnings
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import timm
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_fscore_support, confusion_matrix
from tqdm.auto import tqdm
from IPython.display import display
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_CUDA = DEVICE.type == 'cuda'
if USE_CUDA:
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision('high')
print('Device:', DEVICE)


## Configuration


In [ ]:
cwd = Path.cwd().resolve()
PROJECT_ROOT = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / 'src' / 'wafer_defect').exists() and (candidate / 'configs').exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise RuntimeError('Could not locate repo root')

EXPERIMENT_ROOT = PROJECT_ROOT / 'experiments/anomaly_detection/patchcore/vit_b16/x64/main'
DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'LSWMD.pkl'
IMAGE_SIZE = 64
VIT_FEATURE_BLOCK = 6
PATCH_EMBED_DIM = 128
MEMORY_BANK_MAX = 400000
TOPK_PATCH_RATIO = 0.1
PATCHCORE_NN_K = 3
SCORE_CHUNK = 512
THRESHOLD_QUANTILE = 0.95
TRAIN_NORMAL_N = 40000
VAL_NORMAL_N = 5000
TEST_NORMAL_N = 5000
TEST_DEFECT_N = 250
BATCH_SIZE = 256
NUM_WORKERS = 0
SEED = 42
RETRAIN = False
RUN_UMAP = False

ARTIFACT_DIR = EXPERIMENT_ROOT / 'artifacts'
CHECKPOINTS_DIR = ARTIFACT_DIR / 'checkpoints'
PLOTS_DIR = ARTIFACT_DIR / 'plots'
RESULTS_DIR = ARTIFACT_DIR / 'results'
for directory in [CHECKPOINTS_DIR, PLOTS_DIR, RESULTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f'Image size      : {IMAGE_SIZE}x{IMAGE_SIZE}')
print(f'Patch tokens    : {(IMAGE_SIZE // 16) ** 2} per image')
print(f'ViT block       : {VIT_FEATURE_BLOCK}')
print(f'Artifacts       : {ARTIFACT_DIR}')
print(f'Flags           : RETRAIN={RETRAIN}  RUN_UMAP={RUN_UMAP}')


## Load Dataset


In [ ]:
SRC_ROOT = PROJECT_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))
from wafer_defect.data.legacy_pickle import read_legacy_pickle
df_raw = read_legacy_pickle(DATA_PATH)

def parse_failure_label(v):
    if v is None:
        return 'unknown'
    if isinstance(v, float) and np.isnan(v):
        return 'unknown'
    if isinstance(v, (list, tuple, np.ndarray)):
        a = np.array(v).reshape(-1)
        return 'unknown' if len(a) == 0 else str(a[0])
    return str(v)
df = df_raw.copy()
df['failure_label'] = df['failureType'].apply(parse_failure_label).str.strip()
invalid = {'0', 'unknown', 'nan', 'None', '[]'}
df = df[~df['failure_label'].isin(invalid)].copy()
df['is_anomaly'] = (df['failure_label'].str.lower() != 'none').astype(int)
normal_df = df[df['is_anomaly'] == 0].copy()
defect_df = df[df['is_anomaly'] == 1].copy()
print(f'Labeled: {len(df):,}  Normal: {len(normal_df):,}  Defect: {len(defect_df):,}')
rng = np.random.default_rng(SEED)
ns = normal_df.iloc[rng.permutation(len(normal_df))].reset_index(drop=True)
ds = defect_df.iloc[rng.permutation(len(defect_df))].reset_index(drop=True)
a = TRAIN_NORMAL_N
b = a + VAL_NORMAL_N
c = b + TEST_NORMAL_N
train_normal_df = ns.iloc[0:a].copy()
val_normal_df = ns.iloc[a:b].copy()
test_normal_df = ns.iloc[b:c].copy()
test_defect_df = ds.iloc[0:TEST_DEFECT_N].copy()
del df_raw, df, normal_df, defect_df, ns, ds
gc.collect()
print(f'Train normal : {len(train_normal_df):>7,}')
print(f'Val normal   : {len(val_normal_df):>7,}')
print(f'Test normal  : {len(test_normal_df):>7,}')
print(f'Test defect  : {len(test_defect_df):>7,}')


## Dataset and Model


In [ ]:
class WaferDataset(Dataset):

    def __init__(self, frame: pd.DataFrame, size: int):
        self.maps = frame['waferMap'].values
        self.labels = frame['is_anomaly'].values.astype(np.int64)
        self.size = size

    def __len__(self):
        return len(self.maps)

    def __getitem__(self, idx):
        arr = np.clip(np.array(self.maps[idx], dtype=np.int64), 0, 2)
        x = torch.tensor(arr, dtype=torch.long)
        x = F.one_hot(x, num_classes=3).permute(2, 0, 1).float()
        x = F.interpolate(x.unsqueeze(0), size=(self.size, self.size), mode='nearest').squeeze(0)
        return (x, int(self.labels[idx]))
loader_kw = dict(batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=USE_CUDA, persistent_workers=NUM_WORKERS > 0)
train_loader = DataLoader(WaferDataset(train_normal_df, IMAGE_SIZE), **loader_kw)
val_loader = DataLoader(WaferDataset(val_normal_df, IMAGE_SIZE), **loader_kw)
test_normal_loader = DataLoader(WaferDataset(test_normal_df, IMAGE_SIZE), **loader_kw)
test_defect_loader = DataLoader(WaferDataset(test_defect_df, IMAGE_SIZE), **loader_kw)
xb, _ = next(iter(train_loader))
print(f'Batch shape: {tuple(xb.shape)}')

class ViTPatchExtractor(nn.Module):

    def __init__(self, block_idx=VIT_FEATURE_BLOCK, proj_dim=PATCH_EMBED_DIM):
        super().__init__()
        self.vit = timm.create_model('vit_base_patch16_224.augreg_in21k_ft_in1k', pretrained=True, num_classes=0, img_size=IMAGE_SIZE)
        self._feat = {}
        self.vit.blocks[block_idx].register_forward_hook(lambda m, i, o: self._feat.__setitem__('out', o))
        self.proj = nn.Linear(768, proj_dim, bias=False)

    def forward(self, x):
        self._feat = {}
        self.vit(x)
        toks = self._feat['out'][:, 1:, :]
        return self.proj(toks)
extractor = ViTPatchExtractor().to(DEVICE).eval()
for p in extractor.parameters():
    p.requires_grad = False
with torch.inference_mode():
    dummy = torch.zeros(2, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)
    out = extractor(dummy)
print(f'Extractor output: {tuple(out.shape)}  (B, n_tokens, proj_dim)')


## Build Memory Bank and Score All Splits


In [ ]:
scores_path = RESULTS_DIR / 'scores.npz'
MODEL_EXPORT_PATH = CHECKPOINTS_DIR / 'model.pt'
LEGACY_MODEL_EXPORT_PATH = CHECKPOINTS_DIR / 'best_model.pt'
scores_ready = False

if scores_path.exists() and not RETRAIN:
    checkpoint_path = MODEL_EXPORT_PATH if MODEL_EXPORT_PATH.exists() else LEGACY_MODEL_EXPORT_PATH
    if checkpoint_path.exists():
        checkpoint = torch.load(checkpoint_path, map_location='cpu')
        if isinstance(checkpoint, dict) and 'extractor_state_dict' in checkpoint:
            extractor.load_state_dict(checkpoint['extractor_state_dict'])
            print(f'Loaded extractor weights from {checkpoint_path.name}')
    print(f'Loading saved scores from {scores_path}')
    data = np.load(scores_path)
    val_normal_scores_z = data['val_normal_z']
    test_normal_scores_z = data['test_normal_z']
    test_defect_scores_z = data['test_defect_z']
    mu = float(data['train_mu'])
    std = float(data['train_std'])
    scores_ready = True
elif RETRAIN:
    print('[WARNING] RETRAIN=True remains available for this notebook, but this runtime-fix pass only supports artifact-backed execution.')
else:
    print('[WARNING] RETRAIN is False and the saved scores are missing. Skipping score regeneration.')


## Threshold Selection and Evaluation


In [ ]:
if not (scores_ready):
    print('[WARNING] Saved ViT x64 PatchCore score artifacts are unavailable for this notebook run.')
else:
    threshold_z = float(np.quantile(val_normal_scores_z, THRESHOLD_QUANTILE))
    threshold_raw = mu + threshold_z * std
    print(f'Threshold (z)  : {threshold_z:.4f}')
    print(f'Threshold (raw): {threshold_raw:.6f}')
    y_true = np.concatenate([np.zeros(len(test_normal_scores_z), dtype=int), np.ones(len(test_defect_scores_z), dtype=int)])
    scores = np.concatenate([test_normal_scores_z, test_defect_scores_z])
    y_pred = (scores > threshold_z).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary', zero_division=0)
    auroc = float(roc_auc_score(y_true, scores))
    auprc = float(average_precision_score(y_true, scores))
    cm = confusion_matrix(y_true, y_pred)
    print(f'\nF1   : {f1:.4f}')
    print(f'AUROC: {auroc:.4f}')
    print(f'AUPRC: {auprc:.4f}')
    print(f'Precision: {p:.4f}  Recall: {r:.4f}')
    print(f'Confusion matrix:\n{cm}')
    metrics = dict(f1=float(f1), auroc=float(auroc), auprc=float(auprc), precision=float(p), recall=float(r), threshold_z=float(threshold_z), threshold_raw=float(threshold_raw), train_mu=float(mu), train_std=float(std), confusion_matrix=cm.tolist(), image_size=IMAGE_SIZE, vit_feature_block=VIT_FEATURE_BLOCK, n_patch_tokens=(IMAGE_SIZE // 16) ** 2)
    (RESULTS_DIR / 'evaluation_metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')


## Plots


In [ ]:
if not (scores_ready):
    print('[WARNING] Saved ViT x64 PatchCore score artifacts are unavailable for this notebook run.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(val_normal_scores_z, bins=40, alpha=0.7, color='#457b9d', label='Val normal', density=True)
    axes[0].axvline(threshold_z, color='red', linestyle='--', label=f'={threshold_z:.3f}')
    axes[0].set_title('Val Score Distribution')
    axes[0].set_xlabel('z-score')
    axes[0].set_ylabel('Density')
    axes[0].legend()
    axes[1].hist(test_normal_scores_z, bins=40, alpha=0.6, color='#577590', label='Test normal', density=True)
    axes[1].hist(test_defect_scores_z, bins=40, alpha=0.6, color='#e76f51', label='Test defect', density=True)
    axes[1].axvline(threshold_z, color='red', linestyle='--', label=f'={threshold_z:.3f}')
    axes[1].set_title('Test Score Distribution')
    axes[1].set_xlabel('z-score')
    axes[1].legend()
    plt.suptitle(f'ViT-B/16 PatchCore x64  F1={f1:.3f}  AUROC={auroc:.3f}  AUPRC={auprc:.3f}')
    plt.tight_layout()
    fig.savefig(PLOTS_DIR / 'score_distribution.png', dpi=200, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    fig2, ax2 = plt.subplots(figsize=(4.5, 4))
    im = ax2.imshow(cm, cmap='Blues')
    for (row_i, col_i), val in np.ndenumerate(cm):
        ax2.text(col_i, row_i, str(val), ha='center', va='center', color='black')
    ax2.set_xticks([0, 1], ['pred normal', 'pred anomaly'])
    ax2.set_yticks([0, 1], ['true normal', 'true anomaly'])
    ax2.set_title('Confusion Matrix')
    fig2.colorbar(im, ax=ax2, fraction=0.046, pad=0.04)
    plt.tight_layout()
    fig2.savefig(PLOTS_DIR / 'confusion_matrix.png', dpi=200, bbox_inches='tight')
    plt.show()
    plt.close(fig2)
    print(f'Plots saved to {PLOTS_DIR}')


## Per-Defect Recall Breakdown


In [ ]:
if not (scores_ready):
    print('[WARNING] Saved ViT x64 PatchCore score artifacts are unavailable for this notebook run.')
else:
    tmp = test_defect_df.copy().reset_index(drop=True)
    tmp['score'] = test_defect_scores_z
    tmp['detected'] = (test_defect_scores_z > threshold_z).astype(int)
    defect_df_out = tmp.groupby('failure_label').agg(count=('detected', 'count'), detected=('detected', 'sum'), recall=('detected', 'mean'), mean_score=('score', 'mean')).reset_index().sort_values('recall', ascending=False)
    display(defect_df_out.round(3))
    defect_df_out.to_csv(RESULTS_DIR / 'defect_breakdown.csv', index=False)
    fig3, ax3 = plt.subplots(figsize=(9, 4))
    plot_df = defect_df_out.sort_values('recall', ascending=True)
    ax3.barh(plot_df['failure_label'], plot_df['recall'], color='#81b29a')
    ax3.set_xlim(0, 1)
    ax3.set_title('Per-Defect Recall - ViT-B/16 PatchCore x64')
    ax3.set_xlabel('Recall')
    plt.tight_layout()
    fig3.savefig(PLOTS_DIR / 'defect_breakdown.png', dpi=200, bbox_inches='tight')
    plt.show()
    plt.close(fig3)
    print(f"Defect breakdown saved to {RESULTS_DIR / 'defect_breakdown.csv'}")


## Save Val and Test Score CSVs

Exports score CSVs in the standard format (score, is_anomaly) for downstream ensemble experiments.


In [ ]:
if not (scores_ready):
    print('[WARNING] Saved ViT x64 PatchCore score artifacts are unavailable for this notebook run.')
else:
    pd.DataFrame({'score': val_normal_scores_z, 'is_anomaly': np.zeros(len(val_normal_scores_z), dtype=int)}).to_csv(RESULTS_DIR / 'val_scores.csv', index=False)
    pd.DataFrame({'score': np.concatenate([test_normal_scores_z, test_defect_scores_z]), 'is_anomaly': y_true}).to_csv(RESULTS_DIR / 'test_scores.csv', index=False)
    print('val_scores.csv and test_scores.csv saved to', RESULTS_DIR)


## UMAP Visualization


In [ ]:
import subprocess
from IPython.display import Image as IPImage
from sklearn.decomposition import PCA

UMAP_PNG_PATH = Path(globals().get('UMAP_PNG_PATH', PLOTS_DIR / 'umap_test_embeddings.png'))
UMAP_CSV_PATH = Path(globals().get('UMAP_CSV_PATH', RESULTS_DIR / 'umap_test_embeddings.csv'))
seed_value = int(globals().get('SEED', 42))

if not RUN_UMAP and UMAP_PNG_PATH.exists():
    print(f'Displaying saved UMAP: {UMAP_PNG_PATH}')
    display(IPImage(filename=str(UMAP_PNG_PATH)))
    if UMAP_CSV_PATH.exists():
        display(pd.read_csv(UMAP_CSV_PATH).head())
elif RUN_UMAP:
    if 'test_normal_scores_z' in globals():
        normal_scores = np.asarray(test_normal_scores_z)
        defect_scores = np.asarray(test_defect_scores_z)
        threshold_value = float(globals().get('threshold_z', globals().get('best_thresh')))
        score_column = 'score_z'
    else:
        normal_scores = np.asarray(test_normal_scores)
        defect_scores = np.asarray(test_defect_scores)
        threshold_value = float(globals().get('best_thresh', globals().get('threshold_z')))
        score_column = 'score'

    if 'test_normal_loader' not in globals() or 'test_defect_loader' not in globals():
        loader_batch_size = int(globals().get('BATCH_SIZE', 128))
        loader_workers = int(globals().get('NUM_WORKERS', 0))
        loader_kwargs = dict(
            batch_size=loader_batch_size,
            shuffle=False,
            num_workers=loader_workers,
            pin_memory=USE_CUDA,
            persistent_workers=loader_workers > 0,
        )
        test_normal_loader = DataLoader(WaferDataset(test_normal_df, IMAGE_SIZE), **loader_kwargs)
        test_defect_loader = DataLoader(WaferDataset(test_defect_df, IMAGE_SIZE), **loader_kwargs)

    try:
        import umap.umap_ as umap
    except Exception:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'umap-learn', '-q'])
        import umap.umap_ as umap

    def collect_image_embeddings(loader, frame, split_name, desc):
        embeddings = []
        metadata_rows = []
        seen = 0
        with torch.inference_mode():
            for xb, yb in tqdm(loader, total=len(loader), desc=desc, unit='batch'):
                batch_size = len(yb)
                xb = xb.to(DEVICE)
                with torch.autocast('cuda', torch.float16, enabled=USE_CUDA):
                    feat = extractor(xb)
                    emb = extractor.proj(feat)
                img_emb = F.normalize(emb.float().mean(dim=1), p=2, dim=1)
                embeddings.append(img_emb.cpu().numpy())
                batch_meta = frame.iloc[seen:seen + batch_size][['failure_label', 'is_anomaly']].copy()
                batch_meta = batch_meta.reset_index(drop=True)
                batch_meta['split'] = split_name
                metadata_rows.append(batch_meta)
                seen += batch_size
        return np.concatenate(embeddings, axis=0), pd.concat(metadata_rows, ignore_index=True)

    X_normal, meta_normal = collect_image_embeddings(test_normal_loader, test_normal_df, 'test_normal', 'Embed test-normal')
    X_defect, meta_defect = collect_image_embeddings(test_defect_loader, test_defect_df, 'test_defect', 'Embed test-defect')
    X = np.concatenate([X_normal, X_defect], axis=0)
    umap_df = pd.concat([meta_normal, meta_defect], ignore_index=True)
    umap_df.insert(0, 'point_index', np.arange(len(umap_df)))
    umap_df[score_column] = np.concatenate([normal_scores, defect_scores])
    umap_df['predicted_anomaly'] = (umap_df[score_column] > threshold_value).astype(int)
    umap_df['label_name'] = umap_df['is_anomaly'].map({0: 'normal', 1: 'defect'})

    n_pca = min(32, X.shape[1], X.shape[0] - 1)
    X_pca = PCA(n_components=n_pca, random_state=seed_value).fit_transform(X)
    reducer = umap.UMAP(
        n_components=2,
        n_neighbors=15,
        min_dist=0.1,
        metric='cosine',
        random_state=seed_value,
        transform_seed=seed_value,
        low_memory=True,
    )
    coords = reducer.fit_transform(X_pca)
    umap_df['umap_1'] = coords[:, 0]
    umap_df['umap_2'] = coords[:, 1]

    fig, ax = plt.subplots(figsize=(8, 6))
    normal_mask = umap_df['is_anomaly'].to_numpy() == 0
    defect_mask = ~normal_mask
    ax.scatter(coords[normal_mask, 0], coords[normal_mask, 1], s=8, alpha=0.45, label='normal', c='steelblue', linewidths=0)
    ax.scatter(coords[defect_mask, 0], coords[defect_mask, 1], s=8, alpha=0.6, label='defect', c='tomato', linewidths=0)
    ax.set_title('UMAP of Test Image Embeddings')
    ax.set_xlabel('UMAP-1')
    ax.set_ylabel('UMAP-2')
    ax.legend()
    fig.tight_layout()
    fig.savefig(UMAP_PNG_PATH, dpi=160, bbox_inches='tight')
    plt.show()

    umap_df.to_csv(UMAP_CSV_PATH, index=False)
    print(f'Saved UMAP figure: {UMAP_PNG_PATH}')
    print(f'Saved UMAP points: {UMAP_CSV_PATH}')
else:
    print('[WARNING] RUN_UMAP is False and the saved UMAP artifacts are missing. Skipping UMAP regeneration.')


## Save Outputs


In [ ]:
if not (scores_ready):
    print('[WARNING] Saved ViT x64 PatchCore score artifacts are unavailable for this notebook run.')
else:
    import shutil
    checkpoints_dir = Path(globals().get('CHECKPOINTS_DIR', globals().get('CHKPT_DIR', Path(ARTIFACT_DIR) / 'checkpoints')))
    plots_dir = Path(globals().get('PLOTS_DIR', Path(ARTIFACT_DIR) / 'plots'))
    results_dir = Path(globals().get('RESULTS_DIR', Path(ARTIFACT_DIR) / 'results'))
    checkpoints_dir.mkdir(parents=True, exist_ok=True)
    plots_dir.mkdir(parents=True, exist_ok=True)
    results_dir.mkdir(parents=True, exist_ok=True)
    MODEL_EXPORT_PATH = str(globals().get('MODEL_EXPORT_PATH', checkpoints_dir / 'model.pt'))
    METRICS_EXPORT_PATH = str(globals().get('METRICS_EXPORT_PATH', results_dir / 'evaluation_metrics.json'))
    TEST_SCORES_CSV_PATH = str(globals().get('TEST_SCORES_CSV_PATH', results_dir / 'test_scores.csv'))
    DEFECT_RECALL_CSV_PATH = str(globals().get('DEFECT_RECALL_CSV_PATH', results_dir / 'test_defect_recall.csv'))
    UMAP_PNG_PATH = str(globals().get('UMAP_PNG_PATH', plots_dir / 'umap_test_embeddings.png'))
    UMAP_CSV_PATH = str(globals().get('UMAP_CSV_PATH', results_dir / 'umap_test_embeddings.csv'))
    threshold_value = float(globals().get('threshold_z', globals().get('best_thresh', 0.0)))
    threshold_raw_value = float(globals().get('threshold_raw', threshold_value))
    artifact = {'threshold_z': threshold_value, 'threshold_raw': threshold_raw_value, 'config': {}}
    for key in ['IMAGE_SIZE', 'VIT_FEATURE_BLOCK', 'VIT_FEATURE_BLOCKS', 'PATCH_EMBED_DIM', 'TOPK_PATCH_RATIO', 'PATCHCORE_NN_K']:
        if key in globals():
            artifact['config'][key.lower()] = globals()[key]
    if 'mu' in globals():
        artifact['train_score_mu'] = float(mu)
    if 'std' in globals():
        artifact['train_score_std'] = float(std)
    if 'extractor' in globals():
        artifact['extractor_state_dict'] = extractor.state_dict()
    elif 'model' in globals() and hasattr(model, 'state_dict'):
        artifact['model_state_dict'] = model.state_dict()
    if 'memory_bank' in globals() and memory_bank is not None:
        artifact['memory_bank'] = memory_bank.cpu()
    elif Path(MODEL_EXPORT_PATH).exists():
        try:
            existing_artifact = torch.load(MODEL_EXPORT_PATH, map_location='cpu')
            if isinstance(existing_artifact, dict):
                for key in ['memory_bank', 'extractor_state_dict', 'model_state_dict']:
                    if key in existing_artifact and key not in artifact:
                        artifact[key] = existing_artifact[key]
        except Exception:
            pass
    if artifact:
        torch.save(artifact, MODEL_EXPORT_PATH)
    if 'test_scores_df' in globals():
        test_scores_df.to_csv(TEST_SCORES_CSV_PATH, index=False)
    if 'defect_recall_df' in globals():
        defect_recall_df.to_csv(DEFECT_RECALL_CSV_PATH, index=False)
    elif 'defect_df_out' in globals():
        defect_df_out.rename(columns={'failure_label': 'failure_label'}).to_csv(DEFECT_RECALL_CSV_PATH, index=False)
    elif 'breakdown' in globals():
        breakdown.to_csv(DEFECT_RECALL_CSV_PATH, index=False)
    metrics_payload = {'threshold_z': threshold_value, 'threshold_raw': threshold_raw_value, 'model_export_path': MODEL_EXPORT_PATH, 'test_scores_csv_path': TEST_SCORES_CSV_PATH, 'defect_recall_csv_path': DEFECT_RECALL_CSV_PATH, 'umap_png_path': UMAP_PNG_PATH, 'umap_csv_path': UMAP_CSV_PATH}
    if 'roc_auc' in globals():
        metrics_payload['roc_auc_z'] = float(roc_auc)
    if 'auroc' in globals():
        metrics_payload['roc_auc_z'] = float(auroc)
    if 'cm' in globals():
        metrics_payload['confusion_matrix'] = cm.tolist()
    Path(METRICS_EXPORT_PATH).write_text(json.dumps(metrics_payload, indent=2), encoding='utf-8')
    print(f'Saved model artifact: {MODEL_EXPORT_PATH}')
    print(f'Saved metrics: {METRICS_EXPORT_PATH}')
